# **Store Sales** 🏪

In this notebook, predictive business analytics study is performed on a Kaggle data set which consists of store sales and other relevant information.

The breakdown of the study is as below:

<ul>
<li>Data exploration</li>
<li>Data transformation</li>
<li>Feature selection</li>
<li>Model construction</li>
<li>Performance evaluation</li>
<li>Model improvement</li>
</ul>

Data Source: "https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data"

In [ ]:
import warnings
import numpy as numpy
import pandas as pandas
import matplotlib.pyplot as pyplot
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from scipy.stats import f_oneway
from xgboost import XGBRegressor
from xgboost import cv
import optuna

In [ ]:
warnings.filterwarnings('ignore')
pandas.set_option('display.max_columns', None)
pandas.set_option('display.max_rows', None)

# **Data Exploration**

<ul>
<li>Extract data from each source file</li>
<li>Merge raw data files on appropriate key columns in order to create a main data set</li>
<li>View sample from data set</li>
<li>Get information about data set</li>
<li>Count the number of unique values for each column</li>
<li>Classify variables with the help of business case, viewed sample, data type information and number of unique values</li>
<li>Make descriptive analysis on target variable</li>
</ul>

In [ ]:
Train = pandas.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
Stores = pandas.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
HolidaysEvents = pandas.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
Oil = pandas.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
Transactions = pandas.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
DataSet = pandas.DataFrame()

DataSet = Train
DataSet = pandas.merge(DataSet, Stores, on="store_nbr", how='inner')
DataSet = pandas.merge(DataSet, HolidaysEvents, on="date", how='left')
DataSet = pandas.merge(DataSet, Oil, on="date", how='left')
DataSet = pandas.merge(DataSet, Transactions, on=['date', 'store_nbr'], how='left')
DataSet.rename(columns= {'type_x': 'store_type'}, inplace = True)
DataSet.rename(columns= {'type_y': 'holiday_type'}, inplace = True)

DataSet.sample(5)

In [ ]:
DataSet.info(show_counts=True)

In [ ]:
DataSet.nunique()

In [ ]:
TargetVariable = 'sales'
CategoricalVariables = ['store_nbr', 'family', 'city', 'state', 'store_type', 'cluster', 'holiday_type', 'locale', 'locale_name', 'description']
ContinuousVariables = ['onpromotion', 'dcoilwtico', 'transactions']

In [ ]:
DataSet[TargetVariable].plot(kind='box', title='Boxplot of Target Variable')
DataSet[TargetVariable].describe()

# **Data Transformation**

<ul>
<li>Treat outliers in target variable with the mean value</li>
    <ul>
    <li>Earthquake on April 16, 2016</li>
    <li>Outside of %75 quantile</li>
    </ul>
<li>Fill the missing values with the mean value of each continuous column in data set</li>
<li>Fill the missing values as "None" of each categorical column (Missing values caused by non-holiday days)</li>
<li>Set the data type of categorical variables</li>
<li>Use ordinal encoding to process the values of categorical values to make feature selection and statistical testing properly</li>
<li>View sample from transformed data set</li>
<li>Get information about transformed data set</li>
<li>Make descriptive analysis on target variable to see the effect of outlier treatment</li>
</ul>

In [ ]:
DataSet['sales'][(DataSet['date'] > '2016-04-16') & (DataSet['date'] < '2016-06-16')] = DataSet['sales'].mean()
DataSet['sales'][(DataSet['sales'] > DataSet['sales'].quantile(q=0.75))] = DataSet['sales'].mean()
DataSet['dcoilwtico'].fillna(DataSet['dcoilwtico'].mean(), inplace=True)
DataSet['transactions'].fillna(DataSet['transactions'].mean(), inplace=True)
DataSet['holiday_type'].fillna('None', inplace=True)
DataSet['locale'].fillna('None', inplace=True)
DataSet['locale_name'].fillna('None', inplace=True)
DataSet['description'].fillna('None', inplace=True)
DataSet['transferred'].fillna('None', inplace=True)

for var in CategoricalVariables:
    DataSet[var] = DataSet[var].astype('category')

ordinalEncoder = OrdinalEncoder()
DataSet[CategoricalVariables] = ordinalEncoder.fit_transform(DataSet[CategoricalVariables])

DataSet.sample(5)

In [ ]:
DataSet.info(show_counts=True)

In [ ]:
DataSet[TargetVariable].plot(kind='box', title='Boxplot of Target Variable')
DataSet[TargetVariable].describe()

# **Feature Selection**

Target variable of this study is continuous and there is both categorical and continuous variables among candidate features. During feature selection step, we have to seek for a variation of the pattern between target variable and candidate features. Visualization methods can help us to make investigation however the relationship between variables should be statistically proven.

<ul>
<li> Investigate the relationship between target variable and categorical variables
    <ul>
    <li>Plot a bar chart of each variable for visual evaluation</li>
    <li>Perform ANOVA Test for statistical evaluation</li>
    </ul>
</li>
<li> Investigate the relationship between target variable and continuous variables
    <ul>
    <li>Create a correlation matrix to measure the relationship</li>
    <li>Visualize the correlation matrix to make interpretations</li>
    </ul>
</li>
</ul>

In [ ]:
fig, axs = pyplot.subplots(nrows=5, ncols=2, figsize=(20, 30))

for var, ax in zip(CategoricalVariables, axs.ravel()):
    SalesByCategory = DataSet.groupby(var)[TargetVariable].sum().reset_index()
    sns.barplot(data=SalesByCategory, x=TargetVariable, y=var, ax=ax, orient='h')
    
    AnovaTest = f_oneway(DataSet[TargetVariable], DataSet[var])
    if (AnovaTest[1] < 0.05):
        print(var, "is related with target variable\nP Value:", AnovaTest[1])
    else:
        print(var, "is NOT related with target variable\nP Value:", AnovaTest[1])

pyplot.show()

In [ ]:
CorrelationData = DataSet[ContinuousVariables]
CorrelationData[TargetVariable] = DataSet[TargetVariable]
CorrelationMatrix = CorrelationData.corr()
CorrelationMatrix

pyplot.figure(figsize=(10,5))

mask = numpy.triu(numpy.ones_like(CorrelationMatrix, dtype=bool))

sns.heatmap(CorrelationMatrix,
            cmap='RdBu_r',
            annot=True,
            fmt='.2f',
            vmin=-1, vmax=1)

pyplot.show()

# **XGBoost**

XGBoost is an appropriate algorithm to solve this problem.

<ul>
<li>Set predictor variables after feature selection</li>
<li>Split data set into train and test</li>
<li>Build a base model and make predictions</li>
<li>Evaluate model performance</li>
<li>Perform hyperparameter tuning with Optuna Framework</li>
<li>Rebuild model with best hyperparameters</li>
<li>View sample from actual and predicted values</li>
</ul>

In [ ]:
PredictorVariables = ['store_nbr', 'family', 'city', 'state', 'store_type', 'cluster', 'holiday_type', 'locale', 'locale_name', 'description', 'onpromotion', 'dcoilwtico', 'transactions']

X = DataSet[PredictorVariables]
Y = DataSet[TargetVariable]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state=42)

XGBoostModel = XGBRegressor(enable_categorical=True)

XGBoostModel.fit(X_train, Y_train)

Y_pred = XGBoostModel.predict(X_test)

print("--- Performance ---")
print("Mean Absolute Error: ", metrics.mean_absolute_error(Y_test, Y_pred))

In [ ]:
def objective(trial):
    
    parameters = {
        'learning_rate': trial.suggest_categorical('learning_rate', [0.1, 0.2, 0.3, 0.4, 0.5]),
        'gamma': trial.suggest_float('gamma', 0, 1.1),
        'lambda': trial.suggest_float('lambda', 0, 1.1),
        'alpha': trial.suggest_float('alpha', 0, 1.1),
        'max_depth': trial.suggest_categorical('max_depth', [4, 6, 8, 10]),
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100, 150, 200]),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': trial.suggest_categorical('random_state', [42]),
        'objective': trial.suggest_categorical('objective', ['reg:absoluteerror']),
        'eval_metric': trial.suggest_categorical('eval_metric', ['mae']),
        'gpu_id': trial.suggest_categorical('gpu_id', [1]),
        'enable_categorical': trial.suggest_categorical('enable_categorical', [True])
    }

    optunaModel = XGBRegressor(**parameters)
    
    optunaModel.fit(X_train, Y_train)
    
    Y_pred = optunaModel.predict(X_test)

    return metrics.mean_absolute_error(Y_test, Y_pred)

study = optuna.create_study(study_name='Store_Sales' ,direction='minimize')
study.optimize(objective, n_trials=10)

XGBoostModel = XGBRegressor(**study.best_trial.params)

XGBoostModel.fit(X_train, Y_train)

Y_pred = XGBoostModel.predict(X_test)

print("\nBest Hyperparameters\n", study.best_trial.params)
print("\n--- Performance (After Tuning) ---")
print("Mean Absolute Error: ", metrics.mean_absolute_error(Y_test, Y_pred))

In [ ]:
ResultsDataFrame = pandas.DataFrame({'Actual Data' : Y_test.squeeze(), 'Predicted Data' : Y_pred.squeeze()})
ResultsDataFrame.sample(50)

In [ ]:
Test = pandas.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
SubmissionDataSet = pandas.DataFrame()

SubmissionDataSet = Test
SubmissionDataSet = pandas.merge(SubmissionDataSet, Stores, on="store_nbr", how='inner')
SubmissionDataSet = pandas.merge(SubmissionDataSet, HolidaysEvents, on="date", how='left')
SubmissionDataSet = pandas.merge(SubmissionDataSet, Oil, on="date", how='left')
SubmissionDataSet = pandas.merge(SubmissionDataSet, Transactions, on=['date', 'store_nbr'], how='left')
SubmissionDataSet.rename(columns= {'type_x': 'store_type'}, inplace = True)
SubmissionDataSet.rename(columns= {'type_y': 'holiday_type'}, inplace = True)

SubmissionDataSet['dcoilwtico'].fillna(SubmissionDataSet['dcoilwtico'].mean(), inplace=True)
SubmissionDataSet['transactions'].fillna(SubmissionDataSet['transactions'].mean(), inplace=True)

for var in CategoricalVariables:
    SubmissionDataSet[var] = SubmissionDataSet[var].astype('category')

X_submission = SubmissionDataSet[PredictorVariables]
Y_submission = XGBoostModel.predict(X_submission)

Submission = pandas.DataFrame({'id': SubmissionDataSet.id, 'sales': Y_submission})
Submission.to_csv('Submission.csv', index=False)